In [1]:
# -*- coding: utf-8 -*-
"""
Reddit Sampling Pipeline
--------------------------------------------------------
Builds a Reddit corpus similar to the AARP dataset sampling pattern.
Requires the data to be stored in a folder called 'data'.
Used on the subreddits 

r/changemyview
r/AskReddit
r/TrueReddit
r/AmItheAsshole
r/debate
r/PoliticalDiscussion

This pipeline only looks at the posts itself, the comments have to be matched in a second step. 

"""

"\nReddit Sampling Pipeline\n--------------------------------------------------------\nBuilds a Reddit corpus similar to the AARP dataset sampling pattern.\nRequires the data to be stored in a folder called 'data'.\nUsed on the subreddits \n\nr/changemyview\nr/AskReddit\nr/TrueReddit\nr/AmItheAsshole\nr/debate\nr/PoliticalDiscussion\n\nThis pipeline only looks at the posts itself, the comments have to be matched in a second step. \n\n"

In [2]:
from __future__ import annotations
import hashlib
from typing import Iterable, List, Optional, Tuple, Dict, Union
import numpy as np
import pandas as pd
from pathlib import Path
from glob import glob

In [3]:
# Columns to retain from the original Reddit JSONL files
KEEP_COLS = ["author", "created_utc", "downs", "id", "likes", "num_comments", "ups", "selftext", "title"]

def load_reddit_jsonl_files(
        paths: Union[str, Path, Iterable[Union[str, Path]]]
) -> pd.DataFrame:
    """
    Load one or multiple Reddit JSONL files and return a unified DataFrame.

    Functionality:
    - Reads JSON Lines (.jsonl) files
    - Keeps only predefined columns
    - Converts `created_utc` (Unix timestamp) into a readable datetime format (UTC)
    - Ensures consistent schema across all input files
    """

    # Convert a single path into a list
    if isinstance(paths, (str, Path)):
        paths = [paths]

    dfs: List[pd.DataFrame] = []

    for p in paths:
        p = Path(p)
        if not p.exists():
            print(f"Warning: File not found and skipped: {p}")
            continue

        # Read JSON Lines file
        df = pd.read_json(p, lines=True)

        # Ensure missing expected columns exist (filled with NA)
        missing_cols = [c for c in KEEP_COLS if c not in df.columns]
        for c in missing_cols:
            df[c] = pd.NA

        # Select only the relevant columns
        df = df[KEEP_COLS].copy()

        # Convert Unix timestamp to UTC datetime
        df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s", utc=True)

        dfs.append(df)

    # If no file was successfully processed, return an empty DataFrame with the correct schema
    if not dfs:
        return pd.DataFrame(columns=["id", "author", "created_utc", "ups",
                                     "downs", "likes", "num_comments", "selftext", "title"])

    # Concatenate all loaded DataFrames
    df_all = pd.concat(dfs, ignore_index=True)

    # Order columns in a logical sequence
    df_all = df_all[["id", "author", "created_utc", "ups", "downs", "likes", "num_comments", "selftext", "title"]]

    return df_all


# ==========================================
# usage
# ==========================================

# loading one file
#df_one = load_reddit_jsonl_files("data/r_AskReddit_posts.jsonl")

# for multiple files:

files = glob("data/*_posts.jsonl")
df_all = load_reddit_jsonl_files(files)

# Preview of the loaded data
print(df_all.head())
print(len(df_all))
#print(df_one.dtypes)

        id                author               created_utc  ups  downs  likes  \
0  1dc9fah    wokepatrickbateman 2024-06-10 01:09:09+00:00    0      0    NaN   
1  1dcus8j                rollem 2024-06-10 19:52:43+00:00   67      0    NaN   
2  1dd2imm               Empigee 2024-06-11 01:39:55+00:00   81      0    NaN   
3  1ddds04         wiredmagazine 2024-06-11 13:12:03+00:00  865      0    NaN   
4  1ddlio7  Sensitive_Remove1112 2024-06-11 18:37:33+00:00    1      0    NaN   

   num_comments selftext                                              title  
0             5                                     A Revolution in Biology  
1             9           Nothing bad ever happened when the far right b...  
2             7           A Flattened, Bloody Raccoon on a Highway Isn’t...  
3           148           The Titan Submersible Disaster Shocked the Wor...  
4             2                                           Quarantined Ideas  
159978


In [5]:
# Optional NLP dependencies
_HAS_SBERT = False
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    _HAS_SBERT = True
except Exception:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity


# ------------------------
# Topics
# ------------------------

DEFAULT_TOPICS = [
    "The government should not forgive student loan debt.",
    "Airbnb should be banned in cities.",
    "The federal minimum wage should be increased.",
    "The US should provide financial and military aid to Ukraine.",
    "A universal basic income would kill the economy.",
    "Climate change is one of the greatest threats to humanity.",
    "Fur clothing should be banned.",
    "The government should not invest in renewable energy.",
    "There should only be vegetarian food in cantines.",
    "Gender-neutral language and stating pronouns are silly issues.",
    "Prostitution should be illegal.",
    "Employers should mandate vaccination.",
    "The government should not be responsible for universal health care.",
    "Immigrants should adopt the local language and culture.",
    "We need stricter gun control laws.",
    "The death penalty should be reestablished.",
    "Police officers should wear body cameras.",
    "Artificial Intelligence should replace humans where possible.",
    "Social media is a threat to democracy.",
]


# ------------------------
# Build topic model
# ------------------------

def build_topic_model(topics: List[str]):
    """
    Build topic embeddings (SBERT) or TF-IDF vectors.
    """
    if _HAS_SBERT:
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        topic_embeddings = model.encode(topics, show_progress_bar=False)
        return "sbert", model, topic_embeddings
    else:
        vec = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1, 2))
        topic_matrix = vec.fit_transform(topics)
        return "tfidf", vec, topic_matrix


# ------------------------
# Core filtering function
# ------------------------

def filter_by_topic_similarity(
        df: pd.DataFrame,
        similarity_threshold: float = 0.4,
        topics: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, int, int]:
    """
    Filter dataframe rows based on cosine similarity between `title`
    and a set of topic sentences.

    Requirements:
    - `df` must contain a column `title`.
    - Rows with title == "[removed]" or "[deleted]" are discarded immediately.

    Returns:
    - filtered_df: rows with similarity >= threshold
    - kept: number of rows retained
    - dropped: number of rows removed
    """

    if "title" not in df.columns:
        raise ValueError("Expected column 'title' in dataframe.")

    # Work on a copy
    df = df.copy()

    # Remove [removed] / [deleted] immediately
    mask_valid = ~df["title"].isin(["[removed]", "[deleted]"])
    df = df[mask_valid].copy()
    

    if df.empty:
        return df, 0, 0

    topics = topics or DEFAULT_TOPICS
    texts = df["title"].fillna("").astype(str).tolist()

    # Build topic model
    kind, model, topic_repr = build_topic_model(topics)

    # Compute similarities
    if kind == "sbert":
        text_embeddings = model.encode(texts, show_progress_bar=False)
        sims = cosine_similarity(text_embeddings, topic_repr)

    else:
        # TF-IDF fallback: fit on topics + texts
        vec = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1, 2))
        combined = topics + texts
        X = vec.fit_transform(combined)
        topic_matrix = X[: len(topics), :]
        text_matrix = X[len(topics):, :]
        sims = cosine_similarity(text_matrix, topic_matrix)

    # Determine best topic per row
    best_idx = sims.argmax(axis=1)
    best_sim = sims.max(axis=1)

    df["best_topic_index"] = best_idx
    df["best_topic"] = [topics[i] for i in best_idx]
    df["best_topic_similarity"] = best_sim

    # Threshold filtering
    keep_mask = df["best_topic_similarity"] >= similarity_threshold
    filtered_df = df[keep_mask].copy()

    kept = int(keep_mask.sum())
    dropped = int(len(df) - kept)

    print(f"Comments kept: {kept}")
    print(f"Comments dropped: {dropped}")
    
    return filtered_df, kept, dropped


# ------------------------
# Example usage
# ------------------------

if __name__ == "__main__":
   
    df_posts = df_all
    print(len(df_posts))

    SIMILARITY_THRESHOLD = 0.4
    filtered, kept, dropped = filter_by_topic_similarity(
        df=df_posts,             # your main dataframe from loader
        similarity_threshold=SIMILARITY_THRESHOLD,
        topics=DEFAULT_TOPICS,
    )

    print(filtered.head())
    print(f"Final kept: {kept}, dropped: {dropped}")


159978
Comments kept: 2399
Comments dropped: 157579
          id              author               created_utc  ups  downs  likes  \
67   1do7blq  scientificamerican 2024-06-25 14:27:06+00:00  399      0    NaN   
110  1dc8fau     threequeermeese 2024-06-10 00:17:10+00:00    1      0    NaN   
167  1dc9dbh       Sakuraswifee_ 2024-06-10 01:06:09+00:00    8      0    NaN   
235  1dcazx9       westy81585new 2024-06-10 02:30:49+00:00    1      0    NaN   
537  1dcign3          DemNodules 2024-06-10 10:40:33+00:00  279      0    NaN   

     num_comments                                           selftext  \
67            134                                                      
110             4  I attend a support group for gender diverse ad...   
167             6  my sister 14f , has a stealing problem, she st...   
235             1                                          [removed]   
537           126  I live in a tiny rural town. My friend from Re...   

                            